# Retrieving Adjacent Vertices by Compass Direction

This notebook demonstrates how to retrieve adjacent vertices in specific compass
directions (e.g., North, Southwest, Up) or along custom vectors using **topologic_fast**.

## Concept

In graph-based spatial analysis, it's useful to filter adjacent vertices based on their
relative direction. This is particularly valuable for:

- Building navigation systems ("find rooms to the north")
- Spatial queries in GIS applications
- Architectural circulation analysis
- Directional pathfinding

## Note on topologicpy vs topologic_fast

The original topologicpy has `Graph.AdjacentVerticesByCompassDirection()` and
`Graph.AdjacentVerticesByVector()`. In topologic_fast, we demonstrate how to
implement similar functionality using the available Graph and Vector utilities.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import math

## Helper Functions

First, let's create helper functions for compass directions and graph visualization.

In [ ]:
# Compass direction vectors
COMPASS_DIRECTIONS = {
    'North': [0, 1, 0],
    'Northeast': [1, 1, 0],
    'East': [1, 0, 0],
    'Southeast': [1, -1, 0],
    'South': [0, -1, 0],
    'Southwest': [-1, -1, 0],
    'West': [-1, 0, 0],
    'Northwest': [-1, 1, 0],
    'Up': [0, 0, 1],
    'Down': [0, 0, -1],
}


def normalize_vector(v):
    """Normalize a 3D vector."""
    mag = math.sqrt(v[0]**2 + v[1]**2 + v[2]**2)
    if mag < 1e-10:
        return [0, 0, 0]
    return [v[0]/mag, v[1]/mag, v[2]/mag]


def dot_product(v1, v2):
    """Compute dot product of two 3D vectors."""
    return v1[0]*v2[0] + v1[1]*v2[1] + v1[2]*v2[2]


def get_compass_direction(vector):
    """
    Get the compass direction name for a vector.
    
    Returns the closest compass direction based on angle.
    """
    # Handle vertical vectors first
    if abs(vector[2]) > 0.9:  # Mostly vertical
        return 'Up' if vector[2] > 0 else 'Down'
    
    # Project to XY plane
    xy = normalize_vector([vector[0], vector[1], 0])
    
    # Calculate angle from North (positive Y)
    angle = math.atan2(xy[0], xy[1])  # atan2(x, y) gives angle from +Y
    angle_deg = math.degrees(angle)
    if angle_deg < 0:
        angle_deg += 360
    
    # Map to compass direction
    directions = ['North', 'Northeast', 'East', 'Southeast', 
                  'South', 'Southwest', 'West', 'Northwest']
    idx = int(round(angle_deg / 45)) % 8
    return directions[idx]


def adjacent_vertices_by_direction(graph, vertex, direction_vector, tolerance=0.5):
    """
    Get adjacent vertices that lie in the specified direction.
    
    Parameters:
        graph: tf.Graph
        vertex: tf.Vertex - the source vertex
        direction_vector: [x, y, z] - the direction to search
        tolerance: float - minimum dot product (0.5 = within 60 degrees)
    
    Returns:
        List of vertices in the specified direction
    """
    # Get all adjacent vertices
    adjacent = graph.AdjacentVertices(vertex)
    
    # Get source coordinates
    src_coords = vertex.Coordinates()
    
    # Normalize the direction vector
    direction = normalize_vector(direction_vector)
    
    # Filter by direction
    result = []
    for adj in adjacent:
        adj_coords = adj.Coordinates()
        
        # Compute vector from source to adjacent
        vec = [
            adj_coords[0] - src_coords[0],
            adj_coords[1] - src_coords[1],
            adj_coords[2] - src_coords[2]
        ]
        vec = normalize_vector(vec)
        
        # Check if vector aligns with direction (using dot product)
        dp = dot_product(vec, direction)
        if dp >= tolerance:
            result.append(adj)
    
    return result


def adjacent_vertices_by_compass(graph, vertex, compass_direction, tolerance=0.5):
    """
    Get adjacent vertices in a compass direction.
    
    Parameters:
        graph: tf.Graph
        vertex: tf.Vertex
        compass_direction: str - e.g., 'North', 'Southwest', 'Up'
        tolerance: float - minimum dot product
    
    Returns:
        List of vertices in the specified compass direction
    """
    direction = COMPASS_DIRECTIONS.get(compass_direction)
    if direction is None:
        raise ValueError(f"Unknown compass direction: {compass_direction}")
    
    return adjacent_vertices_by_direction(graph, vertex, direction, tolerance)

In [ ]:
def visualize_graph_3d(graph, highlight_vertex=None, highlight_neighbors=None,
                       vertex_size=8, edge_width=2, title='Graph Visualization',
                       show_axes=True, axis_size=10):
    """
    Visualize a graph in 3D with optional highlighting.
    
    Parameters:
        graph: tf.Graph
        highlight_vertex: tf.Vertex - vertex to highlight in green
        highlight_neighbors: List[tf.Vertex] - vertices to highlight in red
    """
    fig = go.Figure()
    
    # Get all vertices and edges
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Create sets for highlighted vertices
    highlight_coords = set()
    if highlight_vertex:
        hv = highlight_vertex.Coordinates()
        highlight_coords.add((round(hv[0], 6), round(hv[1], 6), round(hv[2], 6)))
    
    neighbor_coords = set()
    if highlight_neighbors:
        for n in highlight_neighbors:
            nc = n.Coordinates()
            neighbor_coords.add((round(nc[0], 6), round(nc[1], 6), round(nc[2], 6)))
    
    # Draw edges
    for edge in edges:
        ev = edge.Vertices()
        if len(ev) == 2:
            p1 = ev[0].Coordinates()
            p2 = ev[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='gray', width=edge_width),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Categorize vertices by highlight status
    normal_x, normal_y, normal_z = [], [], []
    source_x, source_y, source_z = [], [], []
    neighbor_x, neighbor_y, neighbor_z = [], [], []
    
    for v in vertices:
        coords = v.Coordinates()
        key = (round(coords[0], 6), round(coords[1], 6), round(coords[2], 6))
        
        if key in highlight_coords:
            source_x.append(coords[0])
            source_y.append(coords[1])
            source_z.append(coords[2])
        elif key in neighbor_coords:
            neighbor_x.append(coords[0])
            neighbor_y.append(coords[1])
            neighbor_z.append(coords[2])
        else:
            normal_x.append(coords[0])
            normal_y.append(coords[1])
            normal_z.append(coords[2])
    
    # Add normal vertices
    if normal_x:
        fig.add_trace(go.Scatter3d(
            x=normal_x, y=normal_y, z=normal_z,
            mode='markers',
            marker=dict(size=vertex_size, color='blue'),
            name='Vertices'
        ))
    
    # Add source vertex (green)
    if source_x:
        fig.add_trace(go.Scatter3d(
            x=source_x, y=source_y, z=source_z,
            mode='markers',
            marker=dict(size=vertex_size * 2, color='green', symbol='diamond'),
            name='Source Vertex'
        ))
    
    # Add neighbor vertices (red)
    if neighbor_x:
        fig.add_trace(go.Scatter3d(
            x=neighbor_x, y=neighbor_y, z=neighbor_z,
            mode='markers',
            marker=dict(size=vertex_size * 1.5, color='red'),
            name='Direction Neighbors'
        ))
    
    # Add coordinate axes
    if show_axes:
        axis_colors = ['red', 'green', 'blue']
        axis_labels = ['X', 'Y (North)', 'Z (Up)']
        
        for i, (color, label) in enumerate(zip(axis_colors, axis_labels)):
            end = [0, 0, 0]
            end[i] = axis_size
            fig.add_trace(go.Scatter3d(
                x=[0, end[0]], y=[0, end[1]], z=[0, end[2]],
                mode='lines+text',
                line=dict(color=color, width=3, dash='dash'),
                text=['', label],
                textposition='top center',
                showlegend=False,
                hoverinfo='skip'
            ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X (East+)',
            yaxis_title='Y (North+)',
            zaxis_title='Z (Up+)'
        ),
        width=900,
        height=700,
        showlegend=True
    )
    
    return fig

## Create a Sample 3D Grid Graph

Let's create a graph from a 3D grid of cells to demonstrate directional queries.

In [ ]:
# Create a grid of cells
def create_grid_cells(nx, ny, nz, cell_size=10.0, gap=0.0):
    """Create a grid of box cells."""
    cells = []
    for iz in range(nz):
        for iy in range(ny):
            for ix in range(nx):
                x = ix * (cell_size + gap)
                y = iy * (cell_size + gap)
                z = iz * (cell_size + gap)
                cell = tf.Cell.Box(x, y, z, cell_size, cell_size, cell_size)
                cells.append(cell)
    return cells


# Create a 3x3x3 grid
cells = create_grid_cells(3, 3, 3, cell_size=10.0)
print(f"Created {len(cells)} cells")

# Create a CellComplex from the cells
cell_complex = tf.CellComplex.ByCells(cells)
print(f"CellComplex: {cell_complex.NumCells()} cells")

# Create a graph from the CellComplex
# The graph represents room connectivity (cells as vertices, shared faces as edges)
graph = tf.Graph.ByTopology(cell_complex)

print(f"\nGraph Properties:")
print(f"  Vertices (cells): {graph.Order()}")
print(f"  Edges (connections): {graph.Size()}")
print(f"  Density: {graph.Density():.3f}")
print(f"  Is Bipartite: {graph.IsBipartite()}")

In [ ]:
# Visualize the graph
fig = visualize_graph_3d(graph, title='3x3x3 Grid Graph')
fig.show()

## Select a Source Vertex

Let's select a vertex near the center and query adjacent vertices by direction.

In [ ]:
# Find a vertex near the origin (center of the grid)
target = tf.Vertex.ByCoordinates(15, 15, 15)  # Center of middle cell
source_vertex = graph.NearestVertex(target)

source_coords = source_vertex.Coordinates()
print(f"Source Vertex: ({source_coords[0]:.1f}, {source_coords[1]:.1f}, {source_coords[2]:.1f})")

# Get all adjacent vertices
all_adjacent = graph.AdjacentVertices(source_vertex)
print(f"Total adjacent vertices: {len(all_adjacent)}")

for adj in all_adjacent:
    adj_coords = adj.Coordinates()
    # Compute direction
    vec = [adj_coords[i] - source_coords[i] for i in range(3)]
    direction = get_compass_direction(vec)
    print(f"  ({adj_coords[0]:.1f}, {adj_coords[1]:.1f}, {adj_coords[2]:.1f}) - {direction}")

## Query by Compass Direction

In [ ]:
# Query for vertices to the North
direction = 'North'
north_neighbors = adjacent_vertices_by_compass(graph, source_vertex, direction, tolerance=0.5)

print(f"Adjacent vertices to the {direction} (tolerance=0.5):")
for n in north_neighbors:
    coords = n.Coordinates()
    print(f"  ({coords[0]:.1f}, {coords[1]:.1f}, {coords[2]:.1f})")
print(f"Total: {len(north_neighbors)}")

In [ ]:
# Visualize North neighbors
fig = visualize_graph_3d(
    graph, 
    highlight_vertex=source_vertex, 
    highlight_neighbors=north_neighbors,
    title=f'Adjacent Vertices to the {direction}'
)
fig.show()

In [ ]:
# Query for vertices going Up
direction = 'Up'
up_neighbors = adjacent_vertices_by_compass(graph, source_vertex, direction, tolerance=0.5)

print(f"Adjacent vertices going {direction} (tolerance=0.5):")
for n in up_neighbors:
    coords = n.Coordinates()
    print(f"  ({coords[0]:.1f}, {coords[1]:.1f}, {coords[2]:.1f})")
print(f"Total: {len(up_neighbors)}")

In [ ]:
# Visualize Up neighbors
fig = visualize_graph_3d(
    graph, 
    highlight_vertex=source_vertex, 
    highlight_neighbors=up_neighbors,
    title=f'Adjacent Vertices Going {direction}'
)
fig.show()

## Query by Custom Vector

In [ ]:
# Query for vertices in a diagonal direction
custom_vector = [1, 1, 1]  # Northeast and Up
diagonal_neighbors = adjacent_vertices_by_direction(graph, source_vertex, custom_vector, tolerance=0.3)

print(f"Adjacent vertices in direction {custom_vector} (tolerance=0.3):")
for n in diagonal_neighbors:
    coords = n.Coordinates()
    print(f"  ({coords[0]:.1f}, {coords[1]:.1f}, {coords[2]:.1f})")
print(f"Total: {len(diagonal_neighbors)}")

In [ ]:
# Visualize diagonal neighbors
fig = visualize_graph_3d(
    graph, 
    highlight_vertex=source_vertex, 
    highlight_neighbors=diagonal_neighbors,
    title=f'Adjacent Vertices in Direction [1, 1, 1] (Northeast-Up)'
)
fig.show()

## Using topologic_fast Vector Utilities

In [ ]:
# Use tf.Vector for direction calculations
print("Cardinal Directions from tf.Vector:")
print(f"  North: {tf.Vector.North()}")
print(f"  South: {tf.Vector.South()}")
print(f"  East:  {tf.Vector.East()}")
print(f"  West:  {tf.Vector.West()}")
print(f"  Up:    {tf.Vector.Up()}")
print(f"  Down:  {tf.Vector.Down()}")

In [ ]:
# Calculate angles using tf.Vector
north = tf.Vector.North()
east = tf.Vector.East()
northeast = tf.Vector.Normalize([1, 1, 0])

print("Angles between directions:")
print(f"  North to East: {tf.Vector.Angle(north, east):.1f} degrees")
print(f"  North to Northeast: {tf.Vector.Angle(north, northeast):.1f} degrees")
print(f"  North to Up: {tf.Vector.Angle(north, tf.Vector.Up()):.1f} degrees")

In [ ]:
# Use compass angle
# Compass angle is measured clockwise from North (positive Y)
print("Compass angles:")
for name, direction in [('North', [0, 1, 0]), ('East', [1, 0, 0]), 
                        ('South', [0, -1, 0]), ('West', [-1, 0, 0]),
                        ('Northeast', [1, 1, 0]), ('Southwest', [-1, -1, 0])]:
    angle = tf.Vector.CompassAngle(direction)
    print(f"  {name}: {angle:.1f} degrees")

## All Compass Directions from One Vertex

In [ ]:
# Query all compass directions from the source vertex
print(f"Source vertex: {source_vertex.Coordinates()}")
print(f"\nNeighbors by compass direction (tolerance=0.5):")
print("=" * 50)

direction_results = {}
for direction_name in COMPASS_DIRECTIONS.keys():
    neighbors = adjacent_vertices_by_compass(graph, source_vertex, direction_name, tolerance=0.5)
    direction_results[direction_name] = neighbors
    if neighbors:
        coords_list = [f"({n.Coordinates()[0]:.0f}, {n.Coordinates()[1]:.0f}, {n.Coordinates()[2]:.0f})" 
                       for n in neighbors]
        print(f"{direction_name:12s}: {', '.join(coords_list)}")
    else:
        print(f"{direction_name:12s}: (none)")

## Interactive Direction Explorer

In [ ]:
# Create visualizations for each compass direction
from plotly.subplots import make_subplots

# Select key directions to display
directions_to_show = ['North', 'South', 'East', 'West', 'Up', 'Down']

for direction in directions_to_show:
    neighbors = direction_results[direction]
    fig = visualize_graph_3d(
        graph,
        highlight_vertex=source_vertex,
        highlight_neighbors=neighbors,
        title=f'{direction}: {len(neighbors)} neighbor(s)'
    )
    # Don't show all - just print info
    print(f"{direction}: {len(neighbors)} neighbor(s)")

In [ ]:
# Show all directional results in one visualization
fig = go.Figure()

# Draw edges
edges = graph.Edges()
for edge in edges:
    ev = edge.Vertices()
    if len(ev) == 2:
        p1 = ev[0].Coordinates()
        p2 = ev[1].Coordinates()
        fig.add_trace(go.Scatter3d(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
            mode='lines',
            line=dict(color='lightgray', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))

# Draw all vertices in gray
vertices = graph.Vertices()
all_x = [v.Coordinates()[0] for v in vertices]
all_y = [v.Coordinates()[1] for v in vertices]
all_z = [v.Coordinates()[2] for v in vertices]
fig.add_trace(go.Scatter3d(
    x=all_x, y=all_y, z=all_z,
    mode='markers',
    marker=dict(size=4, color='gray', opacity=0.5),
    name='All Vertices'
))

# Draw source vertex
src = source_vertex.Coordinates()
fig.add_trace(go.Scatter3d(
    x=[src[0]], y=[src[1]], z=[src[2]],
    mode='markers',
    marker=dict(size=15, color='green', symbol='diamond'),
    name='Source'
))

# Color map for directions
direction_colors = {
    'North': 'blue', 'South': 'cyan',
    'East': 'red', 'West': 'magenta',
    'Up': 'orange', 'Down': 'brown'
}

# Draw neighbors for each direction
for direction, neighbors in direction_results.items():
    if neighbors and direction in direction_colors:
        x = [n.Coordinates()[0] for n in neighbors]
        y = [n.Coordinates()[1] for n in neighbors]
        z = [n.Coordinates()[2] for n in neighbors]
        fig.add_trace(go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=10, color=direction_colors[direction]),
            name=direction
        ))

fig.update_layout(
    title='Adjacent Vertices by Compass Direction',
    scene=dict(
        aspectmode='data',
        xaxis_title='X (East+)',
        yaxis_title='Y (North+)',
        zaxis_title='Z (Up+)'
    ),
    width=900,
    height=700
)

fig.show()

## Pathfinding with Directional Constraints

Let's demonstrate how directional queries can be used for constrained pathfinding.

In [ ]:
def directional_path(graph, start, end, preferred_direction, direction_weight=0.8):
    """
    Find a path that prefers moving in the specified direction.
    
    This is a simple greedy algorithm that prioritizes neighbors
    in the preferred direction.
    """
    path = [start]
    visited = set()
    visited.add(tuple(start.Coordinates()))
    
    end_coords = end.Coordinates()
    current = start
    
    for _ in range(100):  # Max iterations
        current_coords = current.Coordinates()
        
        # Check if we've reached the end
        if (abs(current_coords[0] - end_coords[0]) < 0.1 and
            abs(current_coords[1] - end_coords[1]) < 0.1 and
            abs(current_coords[2] - end_coords[2]) < 0.1):
            break
        
        # Get adjacent vertices
        neighbors = graph.AdjacentVertices(current)
        
        # Filter unvisited
        unvisited = []
        for n in neighbors:
            coords = n.Coordinates()
            if tuple(coords) not in visited:
                unvisited.append(n)
        
        if not unvisited:
            print("No unvisited neighbors - stuck!")
            break
        
        # Score neighbors by direction alignment and distance to goal
        best_score = -float('inf')
        best_neighbor = None
        
        for n in unvisited:
            n_coords = n.Coordinates()
            
            # Direction from current to neighbor
            vec = [n_coords[i] - current_coords[i] for i in range(3)]
            vec_norm = normalize_vector(vec)
            
            # Direction alignment score
            pref_norm = normalize_vector(preferred_direction)
            direction_score = dot_product(vec_norm, pref_norm)
            
            # Distance to goal (negative, so closer is better)
            dist_to_goal = math.sqrt(sum((n_coords[i] - end_coords[i])**2 for i in range(3)))
            
            # Combined score
            score = direction_weight * direction_score - (1 - direction_weight) * dist_to_goal
            
            if score > best_score:
                best_score = score
                best_neighbor = n
        
        if best_neighbor:
            path.append(best_neighbor)
            visited.add(tuple(best_neighbor.Coordinates()))
            current = best_neighbor
        else:
            break
    
    return path

In [ ]:
# Find a path that prefers going North
start_v = graph.NearestVertex(tf.Vertex.ByCoordinates(5, 5, 5))
end_v = graph.NearestVertex(tf.Vertex.ByCoordinates(25, 25, 25))

print(f"Start: {start_v.Coordinates()}")
print(f"End: {end_v.Coordinates()}")

# Direct path using graph's built-in pathfinding
direct_path = graph.Path(start_v, end_v)
if direct_path:
    direct_verts = direct_path.Vertices()
    print(f"\nDirect path length: {len(direct_verts)} vertices")

# Directional path preferring North-East-Up
directional = directional_path(graph, start_v, end_v, [1, 1, 1], direction_weight=0.6)
print(f"Directional path length: {len(directional)} vertices")

## Summary

This notebook demonstrated:

1. **Graph Creation**: Using `tf.Graph.ByTopology()` to create graphs from CellComplex

2. **Compass Directions**: Defining and using compass direction vectors

3. **Directional Queries**: Filtering adjacent vertices by direction using dot product

4. **Vector Utilities**: Using `tf.Vector` for angle and direction calculations

5. **Visualization**: 3D graph visualization with Plotly

### API Differences from topologicpy

| topologicpy | topologic_fast | Notes |
|------------|----------------|-------|
| `Graph.AdjacentVerticesByCompassDirection()` | Not available | Implement using `AdjacentVertices()` + direction filter |
| `Graph.AdjacentVerticesByVector()` | Not available | Implement using `AdjacentVertices()` + dot product |
| `Vector.CompassDirection()` | `tf.Vector.CompassAngle()` | Returns angle, not direction name |
| `Graph.NearestVertex()` | `tf.Graph.NearestVertex()` | Same API |
| `Graph.AdjacentVertices()` | `tf.Graph.AdjacentVertices()` | Same API |

### Applications

- Building navigation ("find rooms to the north")
- Spatial queries in GIS applications
- Architectural circulation analysis
- Directional pathfinding
- Sunlight analysis (find rooms facing south)
- Wind flow analysis